# T5 — Corners and weak discontinuities

**Facts used** (classical; X-12 of `notes/cross_domain_connections.md`,
corrected per LC-4; catalog c11).

1. d'Alembert: $u(x,t) = \tfrac12[f(x-ct) + f(x+ct)]$ for the ideal string with
   initial displacement $f$ and zero velocity. A slope jump in $f$ (a corner)
   travels on the characteristics unchanged - a *weak* discontinuity
   (Hadamard 1903), not a shock (Courant & Friedrichs 1948).
2. On the mass chain $\omega(k) = 2\sqrt{J/m}|\sin(k/2)|$ (c11; Schrödinger 1914),
   group velocity depends on $k$, and the corner disperses.
3. X-12's table: the $\pi$ twist is a class of the bundle (static, conserved
   as $\mathbb{Z}_2$, dies under nothing); the corner is a singularity of the
   solution (moves at $c$, dies under dispersion and damping). Computed here:
   row "dies under dispersion".

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

In [ ]:
N, c = 256, 1.0
def triangle(x, center=N // 2, half=24):
    return max(0.0, 1 - abs(x - center) / half)

def sharpness(u):
    # largest second difference: the corner is where the slope jumps
    return max(abs(u[i + 1] - 2 * u[i] + u[i - 1]) for i in range(1, len(u) - 1))

def dalembert(t):
    return [0.5 * (triangle(x - c * t) + triangle(x + c * t)) for x in range(N)]

def chain(t_end, dt=0.05):
    u = [triangle(x) for x in range(N)]
    v = [0.0] * N
    steps = int(round(t_end / dt))
    for _ in range(steps):                              # leapfrog / Verlet on the linear chain
        a = [u[(i + 1) % N] - 2 * u[i] + u[(i - 1) % N] for i in range(N)]
        v = [v[i] + dt * a[i] for i in range(N)]
        u = [u[i] + dt * v[i] for i in range(N)]
    return u

times = [0, 10, 20, 40, 60]
rows = []
for t in times:
    s_cont, s_chain = sharpness(dalembert(t)), sharpness(chain(t))
    rows.append((t, s_cont, s_chain))
    print(f"t = {t:>3}: corner sharpness  continuum {s_cont:.4f}   chain {s_chain:.4f}")
print(termplot.plot_xy([(x, u) for x, u in enumerate(chain(40))], width=64, height=10,
                       title="chain displacement at t = 40 (the two half-corners, blurred)", xlabel="site"))
print(termplot.plot_xy([(x, u) for x, u in enumerate(dalembert(40))], width=64, height=10,
                       title="d'Alembert at t = 40 (two half-corners, sharp)", xlabel="x"))

In [ ]:
def check(chain_keeps_corner=False):
    cont = [s for _, s, _ in rows[1:]]
    ch = [s for _, _, s in rows[1:]]
    continuum_conserved = max(cont) - min(cont) < 1e-9 and abs(cont[0] - rows[0][1] / 2) < 1e-9
    chain_decays = all(b < a for a, b in zip(ch, ch[1:])) and ch[-1] < 0.5 * cont[-1]
    return continuum_conserved and (chain_decays != chain_keeps_corner)

falsify(check, {"chain-is-non-dispersive": lambda: {"chain_keeps_corner": True}})

## The reason: group velocity on the chain is not constant

In [ ]:
def vg(k, J=1.0, m=1.0):
    return math.sqrt(J / m) * math.cos(k / 2) * (1 if k >= 0 else -1)

ks = [i * math.pi / 16 for i in range(1, 16)]
print("k/pi  v_group(chain)  v_group(continuum)")
for k in ks[::2]:
    print(f"{k / math.pi:.3f}   {vg(k):.4f}          {c:.4f}")

## Falsifier: c11's continuum mutant must fail

In [ ]:
rc, _ = catalog("c11_chain_dispersion")
assert rc == 0
rc, out = catalog("c11_chain_dispersion", mutant=True)
mutant_must_fail("c11_chain_dispersion", rc, out)